In [2]:
import json
from pathlib import Path

filepath = Path("framework_comparison.jsonl")
records = []
for line in filepath.open():
    records.append(json.loads(line))
len(records)

25

In [4]:
import pandas as pd

df = pd.DataFrame(records)
df.head()
df.columns

Index(['run_id', 'run_name', 'created_at', 'config.dataset',
       'config.framework', 'accuracy', 'macro_avg.f1-score',
       'macro_avg.precision', 'macro_avg.recall', 'macro_avg.support',
       'weighted_avg.f1-score', 'weighted_avg.precision',
       'weighted_avg.recall', 'weighted_avg.support'],
      dtype='object')

In [5]:
cols_to_drop = ["run_id", "run_name", "created_at"]
df = df.drop(columns=cols_to_drop)
df

,config.dataset,config.framework,accuracy,macro_avg.f1-score,macro_avg.precision,macro_avg.recall,macro_avg.support,weighted_avg.f1-score,weighted_avg.precision,weighted_avg.recall,weighted_avg.support
0,DeepPavlov/clinc150,fedot,0.181818,0.002038,0.001204,0.006623,5500,0.055944,0.033058,0.181818,5500
1,DeepPavlov/hwu64,fedot,0.017658,0.000542,0.000276,0.015625,1076,0.000613,0.000312,0.017658,1076
2,DeepPavlov/snips,fedot,0.151429,0.052631,0.449156,0.151429,1400,0.052631,0.449156,0.151429,1400
3,DeepPavlov/minds14,fedot,0.120370,0.080231,0.231848,0.106151,108,0.087636,0.247341,0.120370,108
4,DeepPavlov/massive,fedot,0.072776,0.004057,0.084553,0.018142,2968,0.013965,0.233475,0.072776,2968
5,DeepPavlov/banking77,fedot,0.012987,0.000333,0.000169,0.012987,3080,0.000333,0.000169,0.012987,3080
6,DeepPavlov/clinc150,glueon,0.873455,0.913901,0.876267,0.963700,5500,0.861773,0.894271,0.873455,5500
7,DeepPavlov/hwu64,glueon,0.911710,0.910216,0.908849,0.918053,1076,0.910096,0.913318,0.911710,1076
8,DeepPavlov/snips,glueon,0.990714,0.990721,0.990813,0.990714,1400,0.990721,0.990813,0.990714,1400
9,DeepPavlov/massive,glueon,0.889151,0.877618,0.873620,0.892717,2968,0.888104,0.890377,0.889151,2968


In [18]:
pd.set_option("display.precision", 2)

In [26]:
accuracy_df = pd.pivot_table(df, index="config.framework", columns="config.dataset", values="accuracy")
accuracy_df["average"] = accuracy_df.mean(axis=1)
accuracy_df.sort_values(by="average", ascending=False, inplace=True)
accuracy_df *= 100
accuracy_df.reset_index(inplace=True)
accuracy_df

config.dataset,config.framework,DeepPavlov/banking77,DeepPavlov/clinc150,DeepPavlov/hwu64,DeepPavlov/massive,DeepPavlov/minds14,DeepPavlov/snips,average
0,glueon,93.28,87.35,91.17,88.92,97.22,99.07,92.83
1,h2o,75.32,66.31,77.32,75.30,76.85,98.36,78.24
2,fedot,1.30,18.18,1.77,7.28,12.04,15.14,9.28
3,lama,1.30,18.18,1.77,7.04,8.33,14.50,8.52


In [28]:
print(accuracy_df.to_string(index=False))

config.framework  DeepPavlov/banking77  DeepPavlov/clinc150  DeepPavlov/hwu64  DeepPavlov/massive  DeepPavlov/minds14  DeepPavlov/snips  average
          glueon                 93.28                87.35             91.17               88.92               97.22             99.07    92.83
             h2o                 75.32                66.31             77.32               75.30               76.85             98.36    78.24
           fedot                  1.30                18.18              1.77                7.28               12.04             15.14     9.28
            lama                  1.30                18.18              1.77                7.04                8.33             14.50     8.52


In [30]:
print(accuracy_df["DeepPavlov/clinc150"].to_string(index=False))

87.35
66.31
18.18
18.18


In [24]:
import json
from pathlib import Path

savepath = Path("wandb_oos/AutoML-Eval/results.json")
with savepath.open() as file:
    data = json.load(file)["null"]

In [25]:
cols = ["in_domain_acc", "oos_precision", "oos_recall"]
rows = []
for res in data:
    frame = res["config"].get("framework", None)
    if frame is None:
        continue
    frame = frame["value"]
    metrics = {"framework": frame}
    for metric_name, metrics_dict in res["wandb-summary"].items():
        if metric_name not in cols:
            continue
        metrics[metric_name] = metrics_dict
    rows.append(metrics)
rows

[{'framework': 'gluon',
  'in_domain_acc': 0.9575555555555556,
  'oos_precision': 0.9847094801223242,
  'oos_recall': 0.322},
 {'framework': 'fedot',
  'in_domain_acc': 0,
  'oos_recall': 1,
  'oos_precision': 0.18181818181818182},
 {'framework': 'lama',
  'in_domain_acc': 0,
  'oos_precision': 0.18181818181818182,
  'oos_recall': 1},
 {'framework': 'h2o',
  'oos_precision': 0.8256880733944955,
  'oos_recall': 0.27,
  'in_domain_acc': 0.8522222222222222}]

In [26]:
import pandas as pd

df = pd.DataFrame(rows)
pd.set_option("display.precision", 2)
df[cols] *= 100
print(df.to_string(index=False))

framework  in_domain_acc  oos_precision  oos_recall
    gluon          95.76          98.47        32.2
    fedot           0.00          18.18       100.0
     lama           0.00          18.18       100.0
      h2o          85.22          82.57        27.0


[{'module_name': 'eval-DeepPavlov/clinc150-gluon-gluon-medium-quality-no-hpo',
  'config': {'dataset': {'value': 'DeepPavlov/clinc150'},
   'framework': {'value': 'gluon'}},
  'wandb-summary': {'110.0': {'precision': 0.8529411764705882,
    'recall': 0.9666666666666667,
    'f1-score': 0.90625,
    'support': 30},
   '117.0': {'recall': 1,
    'f1-score': 0.9836065573770492,
    'support': 30,
    'precision': 0.967741935483871},
   '86.0': {'f1-score': 0.7058823529411765,
    'support': 30,
    'precision': 0.631578947368421,
    'recall': 0.8},
   '118.0': {'f1-score': 0.9523809523809523,
    'support': 30,
    'precision': 0.9090909090909091,
    'recall': 1},
   '69.0': {'f1-score': 0.9830508474576272,
    'support': 30,
    'precision': 1,
    'recall': 0.9666666666666667},
   '75.0': {'recall': 1,
    'f1-score': 0.9523809523809523,
    'support': 30,
    'precision': 0.9090909090909091},
   '47.0': {'precision': 0.8333333333333334,
    'recall': 1,
    'f1-score': 0.909090909090